# Limpieza y transformación de datos: Cafe Sales (Dirty Data)

**Materia:** Análisis y visualización de la información  
**Dataset:** [Cafe Sales – Dirty Data for Cleaning Training](https://www.kaggle.com/datasets/ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training) (`dirty_cafe_sales.csv`)

**Objetivo:** aplicar técnicas de limpieza y transformación con Python sobre una base diseñada con problemas de calidad, justificando cada decisión y exportando una versión limpia.

**Estructura:** 1) Carga · 2) Inspección · 3) Limpieza · 4) Recuperación de datos · 5) Faltantes no recuperables · 6) Transformaciones · 7) Validación · 8) Exportación y tabla resumen.

In [1]:
import numpy as np
import pandas as pd

RAW = "dirty_cafe_sales.csv"
PLACEHOLDERS = {"ERROR", "UNKNOWN", ""}
resumen = []

def log(problema, afectados, accion, justificacion):
    """Registra cada problema para construir la tabla resumen al final."""
    resumen.append({"Problema encontrado": problema,
                    "Registros afectados": int(afectados),
                    "Acción realizada": accion,
                    "Justificación": justificacion})

## 1. Carga de la base
Se lee **todo como texto** (`dtype=str`) para no perder marcadores como `ERROR` o `UNKNOWN` y para detectar valores no numéricos antes de convertir tipos.

In [2]:
df = pd.read_csv(RAW, dtype=str)
n_inicial = len(df)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print("Filas x columnas:", df.shape)
df.head()

Filas x columnas: (10000, 8)


,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## 2. Inspección inicial

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   transaction_id    10000 non-null  str  
 1   item              9667 non-null   str  
 2   quantity          9862 non-null   str  
 3   price_per_unit    9821 non-null   str  
 4   total_spent       9827 non-null   str  
 5   payment_method    7421 non-null   str  
 6   location          6735 non-null   str  
 7   transaction_date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [4]:
print("Nulos reales por columna:")
print(df.isna().sum())
print("\nDuplicados exactos:", df.duplicated().sum())
print("IDs repetidos:", df['transaction_id'].duplicated().sum())

Nulos reales por columna:
transaction_id         0
item                 333
quantity             138
price_per_unit       179
total_spent          173
payment_method      2579
location            3265
transaction_date     159
dtype: int64

Duplicados exactos: 0
IDs repetidos: 0


In [5]:
for c in ["item", "payment_method", "location"]:
    print(f"--- {c} ---")
    print(df[c].value_counts(dropna=False), "\n")

--- item ---
item
Juice       1171
Coffee      1165
Salad       1148
Cake        1139
Sandwich    1131
Smoothie    1096
Cookie      1092
Tea         1089
UNKNOWN      344
NaN          333
ERROR        292
Name: count, dtype: int64 

--- payment_method ---
payment_method
NaN               2579
Digital Wallet    2291
Credit Card       2273
Cash              2258
ERROR              306
UNKNOWN            293
Name: count, dtype: int64 

--- location ---
location
NaN         3265
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64 



**Hallazgos de la inspección**
- Todas las columnas son `object` (texto), incluso las numéricas y la fecha.
- Hay nulos reales y además los marcadores de texto `ERROR` y `UNKNOWN` en todas las columnas.
- Las columnas `payment_method` y `location` son las más afectadas.

## 3. Limpieza básica

### 3.1 Espacios sobrantes
Se aplica `strip()` para que `'Coffee '` y `'Coffee'` no cuenten como categorías distintas.

In [6]:
cambiados = 0
for c in df.columns:
    antes = df[c].copy()
    df[c] = df[c].str.strip()
    cambiados += (antes.notna() & (antes != df[c])).sum()
print("Celdas modificadas:", cambiados)
if cambiados:
    log("Espacios sobrantes en textos", cambiados, "strip() en todas las columnas",
        "Evita que variantes del mismo valor cuenten como categorías distintas.")

Celdas modificadas: 0


### 3.2 Marcadores de dato faltante (`ERROR`, `UNKNOWN`, vacío)
Son marcadores de dato faltante, **no valores reales**. Se convierten a `NaN` para tratar todos los faltantes con un solo criterio. Se registra, por columna, cuántos eran marcadores y cuántos nulos reales.

In [7]:
for c in df.columns:
    es_ph = df[c].str.upper().isin(PLACEHOLDERS)
    n_ph, n_nan = int(es_ph.sum()), int(df[c].isna().sum())
    df.loc[es_ph, c] = np.nan
    if n_ph + n_nan:
        log(f"Faltantes en {c} ({n_ph} 'ERROR'/'UNKNOWN' + {n_nan} nulos)",
            n_ph + n_nan, "Unificar todos como NaN",
            "Son marcadores de dato faltante, no valores reales; un solo criterio simplifica la imputación.")
print(df.isna().sum())

transaction_id         0
item                 969
quantity             479
price_per_unit       533
total_spent          502
payment_method      3178
location            3961
transaction_date     460
dtype: int64


### 3.3 Duplicados e identificador
Una transacción sin ID no se puede rastrear, y un ID repetido cuenta dos veces la misma venta.

In [8]:
n = df["transaction_id"].isna().sum()
df = df.dropna(subset=["transaction_id"])
if n: log("Transaction ID faltante", n, "Eliminar filas", "Sin identificador no se puede rastrear ni deduplicar la transacción.")

n = df.duplicated().sum()
df = df.drop_duplicates()
if n: log("Filas completamente duplicadas", n, "drop_duplicates()", "Una misma transacción contada dos veces infla las ventas.")

n = df.duplicated(subset="transaction_id").sum()
df = df.drop_duplicates(subset="transaction_id", keep="first")
if n: log("Transaction ID repetido", n, "Conservar la primera aparición", "El ID debe ser único.")
print("Filas tras revisar duplicados:", len(df))

Filas tras revisar duplicados: 10000


### 3.4 Conversión de tipos
Se convierten a numérico (`quantity`, `price_per_unit`, `total_spent`) y a fecha (`transaction_date`). Lo no interpretable queda como `NaN`/`NaT`.

In [9]:
for c in ["quantity", "price_per_unit", "total_spent"]:
    antes = df[c].isna().sum()
    df[c] = pd.to_numeric(df[c], errors="coerce")
    nuevos = df[c].isna().sum() - antes
    if nuevos: log(f"Valores no numéricos en {c}", nuevos, "to_numeric(errors='coerce')", "No se pueden usar en cálculos.")

antes = df["transaction_date"].isna().sum()
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
nuevos = df["transaction_date"].isna().sum() - antes
if nuevos: log("Fechas con formato inválido", nuevos, "to_datetime(errors='coerce')", "Una fecha no interpretable equivale a faltante.")

log("Columnas con tipo incorrecto (todo texto)", 4,
    "Cast a numérico (quantity, price_per_unit, total_spent) y datetime (transaction_date)",
    "Permite operar, ordenar y agregar correctamente.")
df.dtypes

transaction_id                 str
item                           str
quantity                   float64
price_per_unit             float64
total_spent                float64
payment_method                 str
location                       str
transaction_date    datetime64[us]
dtype: object

### 3.5 Valores imposibles
Cantidades o precios ≤ 0 y totales negativos no son posibles en una venta real.

In [10]:
imp = ((df["quantity"] <= 0) | (df["price_per_unit"] <= 0) | (df["total_spent"] < 0)).sum()
df.loc[df["quantity"] <= 0, "quantity"] = np.nan
df.loc[df["price_per_unit"] <= 0, "price_per_unit"] = np.nan
df.loc[df["total_spent"] < 0, "total_spent"] = np.nan
print("Valores imposibles encontrados:", imp)
if imp: log("Cantidades/precios/totales <= 0", imp, "Convertir a NaN", "No son posibles en una venta real.")

Valores imposibles encontrados: 0


## 4. Recuperación de datos (sin inventar información)
Se recuperan faltantes **solo cuando existe una relación determinista** entre columnas; no se usan medias ni medianas, porque sesgarían los montos.

### 4.1 Precio a partir del ítem
Cada producto tiene un precio fijo. Se obtiene el menú de precios con la moda por ítem.

In [11]:
precio_item = (df.dropna(subset=["item", "price_per_unit"])
                 .groupby("item")["price_per_unit"].agg(lambda s: s.mode().iloc[0]))
# Verificación: cada ítem debe tener un único precio
print("Precios distintos por ítem:\n", df.groupby("item")["price_per_unit"].nunique())
precio_item

Precios distintos por ítem:
 item
Cake        1
Coffee      1
Cookie      1
Juice       1
Salad       1
Sandwich    1
Smoothie    1
Tea         1
Name: price_per_unit, dtype: int64


item
Cake        3.0
Coffee      2.0
Cookie      1.0
Juice       3.0
Salad       5.0
Sandwich    4.0
Smoothie    4.0
Tea         1.5
Name: price_per_unit, dtype: float64

In [12]:
m = df["price_per_unit"].isna() & df["item"].notna()
df.loc[m, "price_per_unit"] = df.loc[m, "item"].map(precio_item)
rec = m.sum() - df.loc[m, "price_per_unit"].isna().sum()
print("Precios recuperados:", rec)
log("Price Per Unit faltante con Item conocido", rec, "Imputar con el precio del menú para ese ítem",
    "Cada producto tiene precio fijo; la recuperación es determinista.")

Precios recuperados: 479


### 4.2 Relación `total = cantidad × precio`

In [13]:
# Total faltante
m = df["total_spent"].isna() & df["quantity"].notna() & df["price_per_unit"].notna()
df.loc[m, "total_spent"] = df.loc[m, "quantity"] * df.loc[m, "price_per_unit"]
print("Totales recuperados:", m.sum())
log("Total Spent faltante", m.sum(), "Calcular quantity × price_per_unit", "Relación aritmética exacta.")

# Cantidad faltante (solo si el resultado es entero)
m = df["quantity"].isna() & df["total_spent"].notna() & df["price_per_unit"].notna()
q = df.loc[m, "total_spent"] / df.loc[m, "price_per_unit"]
ok = (q % 1 == 0)
df.loc[q[ok].index, "quantity"] = q[ok]
print("Cantidades recuperadas:", ok.sum())
log("Quantity faltante", ok.sum(), "Calcular total_spent ÷ price_per_unit",
    "Se recupera solo si el resultado es entero.")

# Precio faltante con total y cantidad
m = df["price_per_unit"].isna() & df["total_spent"].notna() & df["quantity"].notna()
df.loc[m, "price_per_unit"] = df.loc[m, "total_spent"] / df.loc[m, "quantity"]
print("Precios recuperados (total ÷ cantidad):", m.sum())
log("Price Per Unit faltante (con total y cantidad)", m.sum(), "Calcular total_spent ÷ quantity", "Relación aritmética exacta.")

Totales recuperados: 479
Cantidades recuperadas: 456
Precios recuperados (total ÷ cantidad): 48


### 4.3 Ítem a partir del precio
Solo si el precio corresponde a **un único** producto. Precios compartidos (3.0 → Cake/Juice; 4.0 → Sandwich/Smoothie) son ambiguos y **no se imputan**.

In [14]:
precio_a_item = precio_item.reset_index().groupby("price_per_unit")["item"].agg(list)
print(precio_a_item)
unicos = precio_a_item[precio_a_item.str.len() == 1].str[0]
m = df["item"].isna() & df["price_per_unit"].notna()
nuevo = df.loc[m, "price_per_unit"].map(unicos)
df.loc[m, "item"] = nuevo
print("Ítems recuperados:", nuevo.notna().sum())
log("Item faltante con precio único en el menú", nuevo.notna().sum(), "Inferir ítem desde el precio",
    "Solo cuando el precio identifica un único producto; los precios ambiguos no se imputan.")

price_per_unit
1.0                [Cookie]
1.5                   [Tea]
2.0                [Coffee]
3.0           [Cake, Juice]
4.0    [Sandwich, Smoothie]
5.0                 [Salad]
Name: item, dtype: object


Ítems recuperados: 489


### 4.4 Consistencia final
Si las tres columnas existen pero no cuadran, se recalcula el total con la fórmula.

In [15]:
inc = (df[["quantity","price_per_unit","total_spent"]].notna().all(axis=1)
       & ~np.isclose(df["quantity"]*df["price_per_unit"], df["total_spent"]))
print("Registros inconsistentes:", inc.sum())
df.loc[inc, "total_spent"] = df.loc[inc, "quantity"] * df.loc[inc, "price_per_unit"]
if inc.sum(): log("Total ≠ cantidad × precio", inc.sum(), "Recalcular total", "La fórmula es la fuente más confiable.")

Registros inconsistentes: 0


## 5. Faltantes no recuperables
- **Filas sin cantidad, precio o total recuperable:** se eliminan. Imputar con media/mediana distorsionaría las ventas.
- **Categóricas (`item`, `payment_method`, `location`):** se rellenan con `"Unknown"`; la venta tiene monto válido y la categoría desconocida queda explícita.
- **Fecha:** se mantiene `NaT`; inventar fechas distorsionaría series de tiempo.

In [16]:
m = df[["quantity","price_per_unit","total_spent"]].isna().any(axis=1)
log("Filas sin monto/cantidad/precio recuperable", m.sum(), "Eliminar filas",
    "Sin variables numéricas clave no aportan al análisis; imputar con media/mediana sesgaría los totales.")
df = df[~m]

for c in ["item", "payment_method", "location"]:
    n = df[c].isna().sum()
    df[c] = df[c].fillna("Unknown")
    log(f"{c} faltante sin recuperación posible", n, "Rellenar con 'Unknown'",
        "Se conserva la venta (monto válido) y la categoría desconocida queda explícita.")

n = df["transaction_date"].isna().sum()
log("Fecha faltante", n, "Mantener como NaT", "Inventar fechas distorsionaría series de tiempo; se excluyen solo en análisis temporales.")

## 6. Transformaciones

In [17]:
df["quantity"] = df["quantity"].astype(int)
df["year"] = df["transaction_date"].dt.year.astype("Int64")
df["month"] = df["transaction_date"].dt.month.astype("Int64")
df["weekday"] = df["transaction_date"].dt.day_name()
df = df.reset_index(drop=True)
df.head()

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date,year,month,weekday
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08,2023,9,Friday
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16,2023,5,Tuesday
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19,2023,7,Wednesday
3,TXN_7034554,Salad,2,5.0,10.0,Unknown,Unknown,2023-04-27,2023,4,Thursday
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11,2023,6,Sunday


## 7. Validación del resultado

In [18]:
print("Filas originales:", n_inicial, "| Filas finales:", len(df))
print("\nNulos finales:\n", df.isna().sum())
print("\nID único:", df["transaction_id"].is_unique)
print("cantidad × precio == total en todas las filas:",
      np.allclose(df["quantity"]*df["price_per_unit"], df["total_spent"]))
df.info()

Filas originales: 10000 | Filas finales: 9974

Nulos finales:
 transaction_id        0
item                  0
quantity              0
price_per_unit        0
total_spent           0
payment_method        0
location              0
transaction_date    460
year                460
month               460
weekday             460
dtype: int64

ID único: True


cantidad × precio == total en todas las filas: True
<class 'pandas.DataFrame'>
RangeIndex: 9974 entries, 0 to 9973
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    9974 non-null   str           
 1   item              9974 non-null   str           
 2   quantity          9974 non-null   int64         
 3   price_per_unit    9974 non-null   float64       
 4   total_spent       9974 non-null   float64       
 5   payment_method    9974 non-null   str           
 6   location          9974 non-null   str           
 7   transaction_date  9514 non-null   datetime64[us]
 8   year              9514 non-null   Int64         
 9   month             9514 non-null   Int64         
 10  weekday           9514 non-null   str           
dtypes: Int64(2), datetime64[us](1), float64(2), int64(1), str(5)
memory usage: 876.8 KB


In [19]:
for c in ["item", "payment_method", "location"]:
    print(df[c].value_counts(), "\n")

item
Coffee      1286
Salad       1270
Cookie      1211
Tea         1206
Juice       1168
Cake        1138
Sandwich    1127
Smoothie    1094
Unknown      474
Name: count, dtype: int64 

payment_method
Unknown           3168
Digital Wallet    2284
Credit Card       2268
Cash              2254
Name: count, dtype: int64 

location
Unknown     3952
Takeaway    3016
In-store    3006
Name: count, dtype: int64 



## 8. Exportación y tabla resumen

In [20]:
df.to_csv("cafe_sales_clean.csv", index=False)

tabla = pd.DataFrame(resumen)
tabla = tabla[tabla["Registros afectados"] > 0].reset_index(drop=True)
tabla.to_csv("tabla_resumen.csv", index=False)

with open("tabla_resumen.md", "w", encoding="utf-8") as f:
    f.write("| Problema encontrado | Registros afectados | Acción realizada | Justificación |\n|---|---|---|---|\n")
    for _, r in tabla.iterrows():
        f.write(f"| {r['Problema encontrado']} | {r['Registros afectados']} | {r['Acción realizada']} | {r['Justificación']} |\n")

pd.set_option("display.max_colwidth", None)
tabla

,Problema encontrado,Registros afectados,Acción realizada,Justificación
0,Faltantes en item (636 'ERROR'/'UNKNOWN' + 333 nulos),969,Unificar todos como NaN,"Son marcadores de dato faltante, no valores reales; un solo criterio simplifica la imputación."
1,Faltantes en quantity (341 'ERROR'/'UNKNOWN' + 138 nulos),479,Unificar todos como NaN,"Son marcadores de dato faltante, no valores reales; un solo criterio simplifica la imputación."
2,Faltantes en price_per_unit (354 'ERROR'/'UNKNOWN' + 179 nulos),533,Unificar todos como NaN,"Son marcadores de dato faltante, no valores reales; un solo criterio simplifica la imputación."
3,Faltantes en total_spent (329 'ERROR'/'UNKNOWN' + 173 nulos),502,Unificar todos como NaN,"Son marcadores de dato faltante, no valores reales; un solo criterio simplifica la imputación."
4,Faltantes en payment_method (599 'ERROR'/'UNKNOWN' + 2579 nulos),3178,Unificar todos como NaN,"Son marcadores de dato faltante, no valores reales; un solo criterio simplifica la imputación."
5,Faltantes en location (696 'ERROR'/'UNKNOWN' + 3265 nulos),3961,Unificar todos como NaN,"Son marcadores de dato faltante, no valores reales; un solo criterio simplifica la imputación."
6,Faltantes en transaction_date (301 'ERROR'/'UNKNOWN' + 159 nulos),460,Unificar todos como NaN,"Son marcadores de dato faltante, no valores reales; un solo criterio simplifica la imputación."
7,Columnas con tipo incorrecto (todo texto),4,"Cast a numérico (quantity, price_per_unit, total_spent) y datetime (transaction_date)","Permite operar, ordenar y agregar correctamente."
8,Price Per Unit faltante con Item conocido,479,Imputar con el precio del menú para ese ítem,Cada producto tiene precio fijo; la recuperación es determinista.
9,Total Spent faltante,479,Calcular quantity × price_per_unit,Relación aritmética exacta.


## Conclusiones
- La base original (10,000 filas) tenía faltantes en todas las columnas, tanto nulos reales como marcadores `ERROR`/`UNKNOWN`, y todos los tipos estaban como texto.
- Gran parte de los faltantes numéricos se **recuperó con reglas deterministas** (precio fijo por producto y `total = cantidad × precio`), evitando imputaciones estadísticas que sesgarían las ventas.
- Solo se eliminó un pequeño número de filas sin información numérica recuperable; las categorías no recuperables se conservaron como `Unknown`.
- Limitación: `payment_method` y `location` tienen muchos `Unknown`, por lo que los análisis por esas variables deben interpretarse con cautela.